## Top-3 Ranked Predictions Evaluation
### Model: `openai_gptoss120b_v1_ext_top3.csv` | Visit: `Visit_1`

Evaluates ranked drug name output (rank_1 / rank_2 / rank_3) against ground truth active drugs.  
Metrics: Top-1 / Top-2 / Top-3 accuracy, MRR, Jaccard, Precision, Recall, F1, Exact match.

In [2]:
import json
import pandas as pd
import numpy as np
from collections import Counter

# ── Config ────────────────────────────────────────────────────────
PRED_CSV  = 'drug/all_drugs/openai_gptoss120b_v3_ext_top3.csv'
GOLD_JSON = 'drug_ground_truth.json'
VISIT     = 'Visit_3'

ALL_DRUGS = sorted([
    "carbamazepine", "clobazam", "clonazepam", "ethosuximide",
    "lamotrigine", "levetiracetam", "phenobarbital", "phenytoin",
    "topiramate", "valproate",
])

# ── Load ──────────────────────────────────────────────────────────
df = pd.read_csv(PRED_CSV)
with open(GOLD_JSON) as f:
    gold_raw = json.load(f)

# Gold: active drugs per patient for the target visit
gold_drugs = {}
for pid, visits in gold_raw.items():
    v = visits.get(VISIT, {})
    if v:
        gold_drugs[pid] = set(d for d, val in v.get('drug_binary', {}).items() if val == 1)

# Predictions: ranked list of 3 drug names per patient
pred_ranked = {}
for _, row in df.iterrows():
    pid = str(row['patient_id'])
    pred_ranked[pid] = [str(row[f'rank_{k}']).strip().lower() for k in (1, 2, 3)]

common_pids = sorted(set(pred_ranked) & set(gold_drugs))
N = len(common_pids)

print(f"Evaluating {N} patients | Visit: {VISIT}")
print(f"CSV: {PRED_CSV}")

# ── Gold drug frequency ───────────────────────────────────────────
gold_cnt = Counter(d for pid in common_pids for d in gold_drugs[pid])
print(f"\nGold active-drug frequencies (n={N}):")
for d in ALL_DRUGS:
    print(f"  {d:18s}: {gold_cnt[d]:3d}  ({gold_cnt[d]/N*100:4.1f}%)")

# Mean drugs per patient in gold
gold_sizes = [len(gold_drugs[pid]) for pid in common_pids]
print(f"\n  Mean active drugs/patient (gold): {np.mean(gold_sizes):.2f}")

# ── Core metrics ──────────────────────────────────────────────────
top1, top2, top3, mrrs, jaccards, precs, recs, f1s = [], [], [], [], [], [], [], []

for pid in common_pids:
    gold  = gold_drugs[pid]
    preds = pred_ranked[pid]
    pred_set = set(preds)

    top1.append(int(preds[0] in gold))
    top2.append(int(bool(set(preds[:2]) & gold)))
    top3.append(int(bool(pred_set & gold)))

    union = pred_set | gold
    jaccards.append(len(pred_set & gold) / len(union) if union else 1.0)

    tp   = len(pred_set & gold)
    prec = tp / len(pred_set) if pred_set else 0.0
    rec  = tp / len(gold)     if gold      else 1.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    precs.append(prec); recs.append(rec); f1s.append(f1)

    rr = next((1.0 / k for k, d in enumerate(preds, 1) if d in gold), 0.0)
    mrrs.append(rr)

exact = sum(set(pred_ranked[pid]) == gold_drugs[pid] for pid in common_pids)

print(f"\n{'═'*52}")
print(f"  {'Metric':<20}  {'Score':>8}  {'Count':>10}")
print(f"  {'─'*20}  {'─'*8}  {'─'*10}")
print(f"  {'Top-1 Accuracy':<20}  {np.mean(top1):>8.3f}  {sum(top1):>3d}/{N}")
print(f"  {'Top-2 Accuracy':<20}  {np.mean(top2):>8.3f}  {sum(top2):>3d}/{N}")
print(f"  {'Top-3 Accuracy':<20}  {np.mean(top3):>8.3f}  {sum(top3):>3d}/{N}")
print(f"  {'MRR':<20}  {np.mean(mrrs):>8.3f}")
print(f"  {'─'*20}  {'─'*8}  {'─'*10}")
print(f"  {'Jaccard':<20}  {np.mean(jaccards):>8.3f}")
print(f"  {'Precision':<20}  {np.mean(precs):>8.3f}")
print(f"  {'Recall':<20}  {np.mean(recs):>8.3f}")
print(f"  {'F1':<20}  {np.mean(f1s):>8.3f}")
print(f"  {'─'*20}  {'─'*8}  {'─'*10}")
print(f"  {'Exact Match':<20}  {exact/N:>8.3f}  {exact:>3d}/{N}")
print(f"{'═'*52}")

# Store for next cell
_common_pids = common_pids
_gold_drugs  = gold_drugs
_pred_ranked = pred_ranked
_gold_cnt    = gold_cnt
_N           = N

Evaluating 332 patients | Visit: Visit_3
CSV: drug/all_drugs/openai_gptoss120b_v3_ext_top3.csv

Gold active-drug frequencies (n=332):
  carbamazepine     : 178  (53.6%)
  clobazam          :  22  ( 6.6%)
  clonazepam        :  12  ( 3.6%)
  ethosuximide      :   4  ( 1.2%)
  lamotrigine       :   9  ( 2.7%)
  levetiracetam     :  29  ( 8.7%)
  phenobarbital     :  19  ( 5.7%)
  phenytoin         :   3  ( 0.9%)
  topiramate        :   8  ( 2.4%)
  valproate         : 212  (63.9%)

  Mean active drugs/patient (gold): 1.49

════════════════════════════════════════════════════
  Metric                   Score       Count
  ────────────────────  ────────  ──────────
  Top-1 Accuracy           0.877  291/332
  Top-2 Accuracy           0.904  300/332
  Top-3 Accuracy           0.925  307/332
  MRR                      0.897
  ────────────────────  ────────  ──────────
  Jaccard                  0.344
  Precision                0.374
  Recall                   0.808
  F1                       

In [3]:
# ── Per-drug breakdown ────────────────────────────────────────────
common_pids = _common_pids
gold_drugs  = _gold_drugs
pred_ranked = _pred_ranked
gold_cnt    = _gold_cnt
N           = _N

# Rank distribution per drug
print(f"Per-drug Prediction Frequency vs Ground Truth (n={N}):")
print(f"  {'Drug':18s}  {'Gold':>5}  {'Rank1':>5}  {'Rank2':>5}  {'Rank3':>5}  {'AnyTop3':>7}")
print(f"  {'─'*18}  {'─'*5}  {'─'*5}  {'─'*5}  {'─'*5}  {'─'*7}")
for d in ALL_DRUGS:
    g    = gold_cnt[d]
    r1   = sum(1 for pid in common_pids if pred_ranked[pid][0] == d)
    r2   = sum(1 for pid in common_pids if pred_ranked[pid][1] == d)
    r3   = sum(1 for pid in common_pids if pred_ranked[pid][2] == d)
    any3 = sum(1 for pid in common_pids if d in pred_ranked[pid])
    print(f"  {d:18s}  {g:5d}  {r1:5d}  {r2:5d}  {r3:5d}  {any3:7d}")

# Per-drug precision / recall / F1 (treating top-3 set as the prediction)
print(f"\nPer-drug Precision / Recall / F1 (Top-3 set as prediction):")
print(f"  {'Drug':18s}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'TP':>4}  {'FP':>4}  {'FN':>4}")
print(f"  {'─'*18}  {'─'*6}  {'─'*6}  {'─'*6}  {'─'*4}  {'─'*4}  {'─'*4}")
for d in ALL_DRUGS:
    tp = sum(1 for pid in common_pids if d in pred_ranked[pid] and d in gold_drugs[pid])
    fp = sum(1 for pid in common_pids if d in pred_ranked[pid] and d not in gold_drugs[pid])
    fn = sum(1 for pid in common_pids if d not in pred_ranked[pid] and d in gold_drugs[pid])
    pr = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rc = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else 0.0
    print(f"  {d:18s}  {pr:6.3f}  {rc:6.3f}  {f1:6.3f}  {tp:4d}  {fp:4d}  {fn:4d}")

# ── Rank-1 distribution ───────────────────────────────────────────
r1_dist = Counter(pred_ranked[pid][0] for pid in common_pids)
print(f"\nRank-1 Choice Distribution:")
for d, cnt in r1_dist.most_common():
    print(f"  {d:18s}: {cnt:4d}  ({cnt/N*100:5.1f}%)")

# ── Monotherapy analysis ──────────────────────────────────────────
mono_gold = [pid for pid in common_pids if len(gold_drugs[pid]) == 1]
mono_top1 = sum(1 for pid in mono_gold if list(gold_drugs[pid])[0] == pred_ranked[pid][0])
mono_top3 = sum(1 for pid in mono_gold if list(gold_drugs[pid])[0] in pred_ranked[pid])
print(f"\nMonotherapy patients (gold has exactly 1 drug): {len(mono_gold)}")
if mono_gold:
    print(f"  Top-1 hit: {mono_top1}/{len(mono_gold)} = {mono_top1/len(mono_gold):.3f}")
    print(f"  Top-3 hit: {mono_top3}/{len(mono_gold)} = {mono_top3/len(mono_gold):.3f}")

# ── Polytherapy analysis ──────────────────────────────────────────
poly_gold = [pid for pid in common_pids if len(gold_drugs[pid]) > 1]
if poly_gold:
    poly_top3 = sum(1 for pid in poly_gold if bool(set(pred_ranked[pid]) & gold_drugs[pid]))
    print(f"Polytherapy patients (gold has >1 drug): {len(poly_gold)}")
    print(f"  Any-top-3 hit: {poly_top3}/{len(poly_gold)} = {poly_top3/len(poly_gold):.3f}")

# ── Miss analysis: what was the most missed drug? ─────────────────
miss_cnt = Counter()
for pid in common_pids:
    missed = gold_drugs[pid] - set(pred_ranked[pid])
    for d in missed:
        miss_cnt[d] += 1
print(f"\nMost-missed drugs (in gold but not in top-3 prediction):")
for d, cnt in miss_cnt.most_common():
    print(f"  {d:18s}: missed {cnt:3d} times  ({cnt/N*100:.1f}% of patients)")

Per-drug Prediction Frequency vs Ground Truth (n=332):
  Drug                 Gold  Rank1  Rank2  Rank3  AnyTop3
  ──────────────────  ─────  ─────  ─────  ─────  ───────
  carbamazepine         178    124     20      8      152
  clobazam               22      0     37     52       89
  clonazepam             12      0      1      1        2
  ethosuximide            4      3      2      0        5
  lamotrigine             9      3     31     25       59
  levetiracetam          29      9    139    125      273
  phenobarbital          19      9     58    102      169
  phenytoin               3      0      1      1        2
  topiramate              8      0      2      4        6
  valproate             212    184     41     14      239

Per-drug Precision / Recall / F1 (Top-3 set as prediction):
  Drug                  Prec     Rec      F1    TP    FP    FN
  ──────────────────  ──────  ──────  ──────  ────  ────  ────
  carbamazepine        0.836   0.713   0.770   127    25    51